# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an interactive environment for loading and exploring the FAIR\u2072 dataset using the `mlcroissant` library, referencing all schema entities by their `@id`.\n\n### Dataset Source\nAccess provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed\n!pip install mlcroissant

## 1. Data Loading\nLoad metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\n# Define the dataset URL\ncroissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'\n\n# Load the dataset\ndataset = mlc.Dataset(croissant_url)\n# Access dataset metadata\nmetadata = dataset.metadata\nprint(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview\nReview the available record sets, their fields, and the corresponding `@id`s.

In [ ]:
# List all record sets and their fields by `@id`\nrecord_sets = []\nprint("Available record sets and fields:")\nfor rs in getattr(metadata, 'recordSets', getattr(metadata, 'recordSet', [])):\n    # Each rs is an object in mlcroissant\n    rid = rs.id\n    record_sets.append(rid)\n    print(f"  RecordSet @id: {rid}")\n    \n    # List all fields and their column ids\n    print(f"    Fields:")\n    for field in getattr(rs, 'fields', getattr(rs, 'field', [])):\n        print(f"      Field @id: {field.id}, name: {getattr(field, 'name', '') if hasattr(field, 'name') else ''}")\n        if hasattr(field, 'columns'):\n            for column in field.columns:\n                print(f"        Column @id: {column.id} (name: {getattr(column, 'name', '')})")

## 3. Data Extraction\nLoad data from a specific record set into a DataFrame for analysis.\nAll `@id`s are referenced exactly as in the Croissant metadata.

In [ ]:
# Example assumes one main record set, update if more are present\nif not record_sets:\n    # If `record_sets` is empty, try to find any record set\n    # This is a fallback: inspect all top-level metadata keys\n    all_rs = getattr(metadata, 'recordSets', getattr(metadata, 'recordSet', []))\n    for rs in all_rs:\n        record_sets.append(rs.id)\n\ndataframes = {}\nfor record_set_id in record_sets:\n    print(f"Loading records for RecordSet @id: {record_set_id}")\n    records = list(dataset.records(record_set=record_set_id))\n    df = pd.DataFrame(records)\n    dataframes[record_set_id] = df\n    print(f"First 3 records for {record_set_id}:")\n    display(df.head(3))\nif record_sets:\n    sample_id = record_sets[0]\n    print(f"Columns in DataFrame for record set {sample_id}:")\n    print(dataframes[sample_id].columns.tolist())

## 4. Exploratory Data Analysis (EDA)\nApply typical EDA procedures, such as filtering records by thresholds, normalizing values, and grouping.\n**Reference all field and column names by their `'@id'`.**

In [ ]:
# Select a record set and one numeric field for analysis\nrecord_set_id = record_sets[0]  # Use the first found record set\ndf = dataframes[record_set_id]\n\n# List candidate numeric columns\nprint("Numeric columns candidates:")\nprint(df.select_dtypes(include=['number']).columns.tolist())\n\n# User selection: for demonstration, let's choose the first numeric column\nif not df.select_dtypes(include=['number']).columns.empty:\n    numeric_field_id = df.select_dtypes(include=['number']).columns[0]\nelse:\n    numeric_field_id = df.columns[0]  # fallback\nprint(f"Chosen numeric field (@id): {numeric_field_id}")\n\nthreshold = 10\nfiltered_df = df[df[numeric_field_id] > threshold]\nprint(f"Filtered records with {numeric_field_id} > {threshold}:")\ndisplay(filtered_df.head())\n\n# Normalize\nfiltered_df = filtered_df.copy()\ncolnorm = f"{numeric_field_id}_normalized"\nfiltered_df[colnorm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()\nprint(f"Normalized values for {numeric_field_id} (first rows):")\ndisplay(filtered_df[[numeric_field_id, colnorm]].head())\n\n# Group by another (likely categorical) field if one exists\ncatcols = df.select_dtypes(include=['object', 'category']).columns\nif catcols.size > 0:\n    group_field_id = catcols[0]\n    print(f"Grouping by field (@id): {group_field_id}")\n    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()\n    print(grouped_df.head())\nelse:\n    group_field_id = None

## 5. Visualization\nVisualize the distribution of a numeric field (referenced by its `@id`) and relationships between fields.

In [ ]:
# Histogram of the normalized numeric field\nplt.figure(figsize=(8, 4))\nfiltered_df[colnorm].hist(bins=30)\nplt.title(f"Histogram of normalized {numeric_field_id}")\nplt.xlabel(colnorm)\nplt.ylabel('Frequency')\nplt.show()\n\n# Boxplot by group if available\nif group_field_id:\n    plt.figure(figsize=(12, 5))\n    filtered_df.boxplot(column=numeric_field_id, by=group_field_id, grid=False)\n    plt.suptitle("")\n    plt.title(f"{numeric_field_id} by {group_field_id}")\n    plt.ylabel(numeric_field_id)\n    plt.xlabel(group_field_id)\n    plt.xticks(rotation=45)\n    plt.show()

## 6. Conclusion\n- This notebook demonstrated how to use `mlcroissant` to load, inspect, and process a FAIR\u2072 dataset with strict referencing via `@id`.\n- You can now extend these analyses to other record sets or fields as required.\n- For further schema exploration and clean dataset integration, always reference record sets, fields, and columns by their Croissant `@id` in your scripts.